# Notebook 03 — Energy Simulation

**Stage A:** Real-data visualisation for H001, H002, and H003  
**Stage B:** LSTM next-hour forecast on the held-out test period  
**Stage C:** Hourly energy-balance simulation with grid-cost calculation

> **Tariff note.** The electricity prices used here are the **hypothetical illustrative scenarios** stored in `data/processed/energy_price_scenarios.csv`. They are *not* observed Ausgrid tariffs. Interpret all cost figures accordingly.

In [ ]:
# ── Setup ────────────────────────────────────────────────────────────────────
import sys, warnings
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np
import pandas as pd

# Make sure project root is on the path
PROJECT_ROOT = Path('.').resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.energy.energy_balance import compute_energy_balance, compute_energy_balance_series
from src.energy.grid import GridCostCalculator
from src.energy.household import load_households
from src.evaluation.metrics import load_baseline_evaluation_metrics, format_evaluation_summary_table
from src.forecasting.predict import EnergyForecaster
from src.utils.seed import set_seed

set_seed(42)
warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 110, 'font.size': 11})
print('Imports OK')

## Stage A — Real-data visualisation

In [ ]:
# ── A1: Load data ────────────────────────────────────────────────────────────
DATA_PATH = PROJECT_ROOT / 'data' / 'processed' / 'three_house_hourly_lstm_ready.csv'
df = pd.read_csv(DATA_PATH, parse_dates=['timestamp'])
df = df.sort_values(['house_id', 'timestamp']).reset_index(drop=True)

households = load_households(PROJECT_ROOT / 'data' / 'processed' / 'customer_id_mapping.csv')
houses = sorted(df['house_id'].unique())

print(f'Dataset shape: {df.shape} | Households: {houses}')
df.head()

In [ ]:
# ── A2: Household metadata ───────────────────────────────────────────────────
mapping_df = pd.DataFrame([h.to_dict() for h in households.values()])
print('Household metadata:')
display(mapping_df)

In [ ]:
# ── A3: Basic statistics per household ──────────────────────────────────────
stats = df.groupby('house_id')[['total_demand_kwh', 'solar_generation_kwh']].agg(
    ['mean', 'std', 'min', 'max']
).round(3)
print('Per-household hourly statistics (kWh):')
display(stats)

In [ ]:
# ── A4: Representative 24-hour profile ───────────────────────────────────────
# Choose 2013-01-15 as a mid-summer representative day with clear solar generation
REP_DAY = '2013-01-15'
day_df = df[df['timestamp'].dt.date.astype(str) == REP_DAY].copy()

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=False)
colors = {'total_demand_kwh': '#e74c3c', 'solar_generation_kwh': '#f39c12'}

for ax, house in zip(axes, houses):
    hd = day_df[day_df['house_id'] == house]
    ax.step(hd['hour'], hd['total_demand_kwh'], where='post',
            color=colors['total_demand_kwh'], label='Demand', linewidth=2)
    ax.fill_between(hd['hour'], hd['total_demand_kwh'], step='post',
                    color=colors['total_demand_kwh'], alpha=0.15)
    ax.step(hd['hour'], hd['solar_generation_kwh'], where='post',
            color=colors['solar_generation_kwh'], label='Solar Generation', linewidth=2)
    ax.fill_between(hd['hour'], hd['solar_generation_kwh'], step='post',
                    color=colors['solar_generation_kwh'], alpha=0.15)
    h_obj = households[house]
    ax.set_title(f'{house} — {h_obj.generator_capacity_kwp} kWp system')
    ax.set_xlabel('Hour of day')
    ax.set_ylabel('Energy (kWh)')
    ax.set_xticks(range(0, 24, 4))
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)

fig.suptitle(f'Representative 24-hour profile — {REP_DAY}', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'data' / 'processed' / '03_daily_profile.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── A5: Show LSTM evaluation metrics (preserved baseline) ────────────────────
metrics = load_baseline_evaluation_metrics(PROJECT_ROOT / 'data' / 'processed' / 'evaluation_metrics.csv')
summary = format_evaluation_summary_table(metrics)
print('Held-out test-period evaluation metrics:')
print('NOTE: Demand LSTM MAE is worse than persistence — this is the documented baseline.\n')
display(summary.rename(columns={
    'target_label': 'Target', 'model_label': 'Model',
    'n_test_hours': 'N', 'mae_kwh': 'MAE (kWh)', 'rmse_kwh': 'RMSE (kWh)'
}))

## Stage B — LSTM next-hour forecast

In [ ]:
# ── B1: Load forecaster and run inference ────────────────────────────────────
FORECAST_TS = '2013-06-08 10:00:00'   # Hour with H001 surplus and H002 deficit
forecaster = EnergyForecaster(
    data_path=PROJECT_ROOT / 'data' / 'processed' / 'three_house_hourly_lstm_ready.csv',
    model_path=PROJECT_ROOT / 'data' / 'processed' / 'lstm_model.keras',
    scaler_path=PROJECT_ROOT / 'data' / 'processed' / 'lstm_scaler.joblib',
)

forecast_results = forecaster.predict_all_households(FORECAST_TS)

rows = []
for h, res in forecast_results.items():
    rows.append(res.to_dict())
forecast_df = pd.DataFrame(rows)
print(f'One-step-ahead LSTM forecasts for {FORECAST_TS}:')
display(forecast_df[[
    'house_id', 'predicted_demand_kwh', 'actual_demand_kwh', 'demand_error_kwh',
    'predicted_solar_generation_kwh', 'actual_solar_generation_kwh', 'solar_error_kwh',
    'is_held_out_period'
]])

In [ ]:
# ── B2: Visualise actual vs predicted for H001 over a rolling 7-day window ──
test_preds = pd.read_csv(
    PROJECT_ROOT / 'data' / 'processed' / 'test_predictions.csv',
    parse_dates=['forecast_timestamp']
)
h1_preds = test_preds[test_preds['house_id'] == 'H001'].sort_values('forecast_timestamp')

# Show one representative week
week = h1_preds[
    (h1_preds['forecast_timestamp'] >= '2013-06-01') &
    (h1_preds['forecast_timestamp'] < '2013-06-08')
]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 6), sharex=True)

ax1.plot(week['forecast_timestamp'], week['actual_demand_kwh'],
         color='#2c3e50', linewidth=1.5, label='Actual')
ax1.plot(week['forecast_timestamp'], week['predicted_demand_kwh'],
         color='#e74c3c', linewidth=1.2, linestyle='--', label='LSTM Predicted')
ax1.set_ylabel('Demand (kWh)'); ax1.legend(); ax1.grid(alpha=0.3)
ax1.set_title('H001 — Actual vs LSTM forecast (one-week test window)')

ax2.plot(week['forecast_timestamp'], week['actual_solar_generation_kwh'],
         color='#2c3e50', linewidth=1.5, label='Actual')
ax2.plot(week['forecast_timestamp'], week['predicted_solar_generation_kwh'],
         color='#f39c12', linewidth=1.2, linestyle='--', label='LSTM Predicted')
ax2.set_ylabel('Solar Generation (kWh)'); ax2.legend(); ax2.grid(alpha=0.3)
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
plt.xticks(rotation=20)
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'data' / 'processed' / '03_forecast_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

## Stage C — Energy-balance simulator

In [ ]:
# ── C1: Compute full test-period energy balances using ACTUAL readings ────────
SCENARIO_ID = 'HYPOTHETICAL_BASE'
grid_calc = GridCostCalculator(PROJECT_ROOT / 'data' / 'processed' / 'energy_price_scenarios.csv')
scenario = grid_calc.get_scenario(SCENARIO_ID)

print(f'⚠️  Tariff used: {scenario.scenario_label}')
print(f'   Import: {scenario.import_price_aud_per_kwh} AUD/kWh | '
      f'Export credit: {scenario.solar_export_credit_aud_per_kwh} AUD/kWh')
print(f'   is_observed_tariff = {scenario.is_observed_tariff}  (These are HYPOTHETICAL illustrative values)\n')

# Compute energy balance across test period using actual readings
test_actual = test_preds[['house_id', 'forecast_timestamp', 'actual_demand_kwh', 'actual_solar_generation_kwh']].copy()
test_actual = compute_energy_balance_series(
    test_actual,
    demand_col='actual_demand_kwh',
    solar_col='actual_solar_generation_kwh',
    timestamp_col='forecast_timestamp'
)

# Also compute with LSTM predictions for comparison
test_predicted = test_preds[['house_id', 'forecast_timestamp', 'predicted_demand_kwh', 'predicted_solar_generation_kwh']].copy()
test_predicted = compute_energy_balance_series(
    test_predicted,
    demand_col='predicted_demand_kwh',
    solar_col='predicted_solar_generation_kwh',
    timestamp_col='forecast_timestamp'
)

print('Test-period energy balance (using actual readings):')
display(test_actual.groupby('house_id')[['actual_demand_kwh', 'actual_solar_generation_kwh',
                                          'self_consumption_kwh', 'deficit_kwh', 'surplus_kwh']].sum().round(2))

In [ ]:
# ── C2: Calculate grid costs per household ───────────────────────────────────
cost_rows = []
for house in houses:
    hdf = test_actual[test_actual['house_id'] == house]
    total_import = hdf['deficit_kwh'].sum()
    total_export = hdf['surplus_kwh'].sum()
    total_self   = hdf['self_consumption_kwh'].sum()
    import_cost  = total_import * scenario.import_price_aud_per_kwh
    export_rev   = total_export * scenario.solar_export_credit_aud_per_kwh
    net_cost     = import_cost - export_rev
    cost_rows.append({
        'house_id': house,
        'total_demand_kwh': round(hdf['actual_demand_kwh'].sum(), 2),
        'total_solar_kwh': round(hdf['actual_solar_generation_kwh'].sum(), 2),
        'self_consumption_kwh': round(total_self, 2),
        'grid_imported_kwh': round(total_import, 2),
        'surplus_exported_kwh': round(total_export, 2),
        'import_cost_aud': round(import_cost, 2),
        'export_credit_aud': round(export_rev, 2),
        'net_grid_cost_aud': round(net_cost, 2),
        'tariff': SCENARIO_ID,
        'is_hypothetical_tariff': True,
    })

cost_df = pd.DataFrame(cost_rows)
print(f'Grid cost summary — test period (1,752 hours per house) | Tariff: {SCENARIO_ID}')
display(cost_df)

In [ ]:
# ── C3: Energy-balance time series plot ──────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(13, 9), sharex=True)

for ax, house in zip(axes, houses):
    hdf = test_actual[test_actual['house_id'] == house].sort_values('forecast_timestamp')
    # Resample to weekly for readability
    hdf_w = hdf.set_index('forecast_timestamp').resample('W')[
        ['actual_demand_kwh', 'actual_solar_generation_kwh', 'deficit_kwh', 'surplus_kwh']
    ].sum()
    ax.fill_between(hdf_w.index, hdf_w['deficit_kwh'], label='Grid Import (Deficit)', color='#e74c3c', alpha=0.7)
    ax.fill_between(hdf_w.index, -hdf_w['surplus_kwh'], label='Grid Export (Surplus)', color='#27ae60', alpha=0.7)
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_ylabel('Energy (kWh/week)')
    ax.set_title(f'{house} — Weekly grid import vs solar export (test period)')
    ax.legend(loc='upper right', fontsize=9)
    ax.grid(alpha=0.25)

axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'data' / 'processed' / '03_energy_balance_timeseries.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── C4: Hourly breakdown at the representative game-theory hour ──────────────
print(f'Hourly snapshot at {FORECAST_TS}:')
for h, res in forecast_results.items():
    b = compute_energy_balance(
        demand_kwh=res.actual_demand_kwh,
        solar_generation_kwh=res.actual_solar_generation_kwh,
        house_id=h,
    )
    gc = grid_calc.calculate_cost(b, scenario_id=SCENARIO_ID)
    print(f'  {h}: demand={b.demand_kwh:.3f} kWh, solar={b.solar_generation_kwh:.3f} kWh, '
          f'deficit={b.deficit_kwh:.3f} kWh, surplus={b.surplus_kwh:.3f} kWh, '
          f'net_cost={gc.net_cost_aud:.4f} AUD')